# ms-building-footprints — does an independent building source agree with OSM?

**Status:** `exploring`

`building` is one of only two classes measured good, and the whole quality story
rests on a single source. Microsoft's Global ML Building Footprints is derived
from Bing aerial imagery by a model — **independent of OSM in both data and
method**. So it can say three things OSM alone cannot:

1. **Does OSM miss buildings?** MS-only footprints over ground OSM left blank.
2. **Is the 2.84 m label offset real?** Two independent maps disagreeing with the
   imagery by the *same* vector is confirmation; disagreeing differently is not.
3. **Can the two be merged** into a denser, better-conditioned `building` class?

### What would make this `failed`

- MS footprints agree with OSM no better than the null baseline → not an
  independent source, just a copy or noise.
- MS-vs-imagery offset is uncorrelated with OSM-vs-imagery offset → the offset is
  not a property of the maps and the per-location correction is unjustified.
- Coverage over rural Ukraine is so sparse it adds nothing OSM did not have.

Run from `src/` with the `uavloc` env.
`papermill pocs/ms-building-footprints/probe.ipynb pocs/ms-building-footprints/runs/probe-$(date +%F).ipynb -p n 59`

In [ ]:
# papermill parameters
SLUG = "ms-building-footprints"
n = 9              # locations; 59 = all
links_url = "https://bfppub.blob.core.windows.net/$web/2026-08-13/dataset-links.csv"
country = "Ukraine"

## Setup

In [ ]:
import gzip, io, json, sys, time, urllib.request, warnings
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

sys.path.insert(0, str(Path.cwd() / "pocs" / "map-derived-labels"))
import align, osm                              # noqa: E402
from tiles import TileGrid                     # noqa: E402

ROOT = Path("data/derived/ua-village-multidate")
HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"
FIGS.mkdir(parents=True, exist_ok=True)


def locations():
    return json.loads((ROOT / "index.json").read_text())["built"]


def load_location(name, date="current"):
    """(image, grid, meta) at full resolution — same path build.py rasterised through."""
    meta = json.loads((ROOT / name / "meta.json").read_text())
    img = Image.open(ROOT / name / f"{date}.jpg").convert("RGB")
    return img, TileGrid(**meta["grid"]), meta


def osm_mask(name, cls):
    """One class from the dataset's own label.png, shifted by its measured offset.

    The build recorded `label_offset` per location. Charging the dataset's known
    2.84 m label error to the thing under test would be unfair, so undo it here.
    """
    meta = json.loads((ROOT / name / "meta.json").read_text())
    lab = np.array(Image.open(ROOT / name / "label.png"))
    m = lab == osm.CLASSES[cls]
    off = meta.get("label_offset") or {}
    dx, dy = int(off.get("dx_px", 0)), int(off.get("dy_px", 0))
    if dx or dy:
        m = np.roll(np.roll(m, dy, axis=0), dx, axis=1)
    return m


def iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else float("nan")


def null_iou(pred, truth, k=20, seed=0):
    """Same mask, position destroyed. Anything not beating this is an artifact
    of how much ground got painted (bug-016)."""
    rng = np.random.default_rng(seed)
    h, w = pred.shape
    return float(np.median([
        iou(np.roll(np.roll(pred, rng.integers(h // 8, h - h // 8), 0),
                    rng.integers(w // 8, w - w // 8), 1), truth)
        for _ in range(k)
    ]))

## The source

MS publishes one index CSV of `(Location, QuadKey, Url, Size)`. QuadKeys are
z9 tiles; each file is `.csv.gz` whose contents are **geojsonl**, not CSV —
the extension lies, and that is the single thing most likely to break this cell.
Licence is CDLA Permissive 2.0.

Verify the column names and the payload shape before trusting anything below.

In [ ]:
CACHE = Path("data/derived/ms-buildings")
CACHE.mkdir(parents=True, exist_ok=True)


def get(url, dest=None):
    if dest and dest.exists():
        return dest.read_bytes()
    req = urllib.request.Request(url, headers={"User-Agent": "uav-localization-research/0.1"})
    with urllib.request.urlopen(req, timeout=180) as r:
        blob = r.read()
    if dest:
        dest.write_bytes(blob)
    return blob


raw = get(links_url, CACHE / "dataset-links.csv").decode("utf-8", "replace")
header, *rows = raw.strip().split("\n")
cols = [c.strip() for c in header.split(",")]
print("columns:", cols)

recs = [dict(zip(cols, r.split(","))) for r in rows]
ua = [r for r in recs if r.get("Location", "").strip() == country]
print(f"{len(recs)} rows total, {len(ua)} for {country}")
assert ua, f"no rows for {country} — check the Location spelling in `cols` above"
print("sample:", ua[0])

In [ ]:
def quadkey(lat, lon, z=9):
    """Bing quadkey for the z-level tile containing (lat, lon)."""
    import math
    lat = max(min(lat, 85.05112878), -85.05112878)
    n = 1 << z
    x = int((lon + 180.0) / 360.0 * n)
    s = math.sin(math.radians(lat))
    y = int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
    x, y = min(x, n - 1), min(y, n - 1)
    key = ""
    for i in range(z, 0, -1):
        d, mask = 0, 1 << (i - 1)
        if x & mask:
            d += 1
        if y & mask:
            d += 2
        key += str(d)
    return key


names = locations()[:n]

# A z9 quadkey is ~50 km across at this latitude and a location box is 1.28 km,
# so most locations sit inside one tile -- but not all. Measured: 3 of the 59
# span two quadkeys. Taking only the centre's key silently drops the buildings
# on the far side, which would look exactly like "OSM has buildings MS does not"
# -- the very thing this probe is testing. So union all four corners.
keys_for = {}
for name in names:
    meta = json.loads((ROOT / name / "meta.json").read_text())
    b = meta["bounds"]
    keys_for[name] = sorted({quadkey(la, lo)
                             for la in (b["top"], b["bottom"])
                             for lo in (b["left"], b["right"])})

need = sorted({k for ks in keys_for.values() for k in ks})
spanning = {nm: ks for nm, ks in keys_for.items() if len(ks) > 1}
print(f"{len(names)} locations -> {len(need)} z9 quadkeys")
print(f"locations spanning >1 quadkey: {len(spanning)}"
      + (f" -> {list(spanning)}" if spanning else ""))

by_key = {r["QuadKey"]: r for r in ua}
missing = [k for k in need if k not in by_key]
print("quadkeys with no MS file:", missing or "none")
assert not missing, "some ground has no MS coverage; handle before scoring"

In [ ]:
def load_quadkey(key):
    """Return the list of GeoJSON features in one MS quadkey file."""
    row = by_key.get(key)
    if row is None:
        return []
    dest = CACHE / f"{key}.csv.gz"
    blob = get(row["Url"], dest)
    feats = []
    with gzip.open(io.BytesIO(blob), "rt", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                feats.append(json.loads(line))
    return feats


t0 = time.time()
tiles_feats = {k: load_quadkey(k) for k in need}
print(f"fetched {len(tiles_feats)} quadkeys in {time.time()-t0:.0f}s")
for k, f in tiles_feats.items():
    users = [nm for nm, ks in keys_for.items() if k in ks]
    print(f"  {k}: {len(f):>8,} footprints  ({len(users)} loc)")


def features_for(name):
    """Every footprint from every quadkey this location touches."""
    out = []
    for k in keys_for[name]:
        out.extend(tiles_feats[k])
    return out

## The control

OSM `building`, from the dataset's own `label.png`, shifted back by the
per-location `label_offset` the build measured. That is the trusted yardstick —
measured good, 4.09% coverage — and it is what makes this probe falsifiable.

Paired with a **null baseline** because the score is an aggregate (`bug-016`).

In [ ]:
def burn(features, grid, shape):
    """Rasterise MS footprints through the tile grid's closed-form georeference.

    TileGrid.to_pixel, not Bounds.to_pixel — Bounds is linear in latitude and
    drifts across the mosaic. Using the grid is what makes these land where the
    dataset's own labels land.
    """
    canvas = Image.new("L", (shape[1], shape[0]), 0)
    draw = ImageDraw.Draw(canvas)
    drawn = 0
    for feat in features:
        geom = feat.get("geometry", {})
        if geom.get("type") != "Polygon":
            continue
        ring = geom["coordinates"][0]
        pts = [grid.to_pixel(lat, lon, shape) for lon, lat in ring]
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        if max(xs) < 0 or min(xs) > shape[1] or max(ys) < 0 or min(ys) > shape[0]:
            continue
        draw.polygon(pts, fill=1)
        drawn += 1
    return np.array(canvas, bool), drawn

## Run

In [ ]:
results = []
for name in names:
    img, grid, meta = load_location(name)
    shape = np.array(img).shape[:2]
    ms, drawn = burn(features_for(name), grid, shape)
    ref = osm_mask(name, "building")

    # Offset against the imagery, measured exactly the way the build measured
    # OSM's: image GRADIENT vs label OUTLINE, not raw image vs filled mask.
    # peak_z < 4 means the two layers share no structure and the offset is
    # meaningless however precise it looks (align.offset docstring).
    grey = np.array(img.convert("L"), float)
    try:
        ms_off = align.offset(align.gradient_magnitude(grey),
                              align.outline(ms), meta["mpp"])
        ms_off["trusted"] = bool(ms_off["peak_z"] >= 4 and not ms_off["at_search_limit"])
    except Exception as exc:                     # noqa: BLE001
        ms_off = {"error": repr(exc)}

    results.append({
        "location": name,
        "ms_drawn": drawn,
        "ms_frac": float(ms.mean()),
        "osm_frac": float(ref.mean()),
        "iou": iou(ms, ref),
        "null_iou": null_iou(ms, ref),
        "ms_only_frac": float((ms & ~ref).mean()),
        "osm_only_frac": float((ref & ~ms).mean()),
        "osm_recall_by_ms": float((ms & ref).sum() / ref.sum()) if ref.sum() else float("nan"),
        "ms_offset": ms_off,
        "osm_offset": meta.get("label_offset"),
        "mpp": meta["mpp"],
    })
    r = results[-1]
    print(f"{name:34s} ms {drawn:>6,}  IoU {r['iou']:.3f}  null {r['null_iou']:.3f}  "
          f"recall {r['osm_recall_by_ms']:.3f}")

(HERE / "results.json").write_text(json.dumps(results, indent=1))

## Numbers

Distribution, never a single extremum (`bug-019`). Three questions, in order:

1. **Is MS independent and real?** IoU vs its own null.
2. **Does it add anything?** `ms_only_frac` against `osm_frac`.
3. **Is the offset a map property?** correlation of the two offset vectors.

In [ ]:
import statistics as st


def col(k):
    return [r[k] for r in results if isinstance(r.get(k), float) and not np.isnan(r[k])]


print("             n   median     min     max")
for k in ("iou", "null_iou", "osm_recall_by_ms", "ms_only_frac", "osm_only_frac"):
    v = col(k)
    print(f"{k:>18s} {len(v):>3}  {st.median(v):.4f}  {min(v):.4f}  {max(v):.4f}")

beat = [r for r in results if r["iou"] > r["null_iou"] * 2]
print(f"\nlocations beating 2x their null: {len(beat)}/{len(results)}")

pairs = [(r["ms_offset"], r["osm_offset"]) for r in results
         if isinstance(r["ms_offset"], dict) and r["ms_offset"].get("trusted")
         and (r["osm_offset"] or {}).get("peak_z", 0) >= 4]
print(f"\nlocations where BOTH offsets have peak_z >= 4: {len(pairs)}/{len(results)}")
if pairs:
    mx = np.array([[p[0]["dx_px"], p[0]["dy_px"]] for p in pairs], float)
    ox = np.array([[p[1]["dx_px"], p[1]["dy_px"]] for p in pairs], float)
    print(f"\noffset agreement over {len(pairs)} locations")
    print(f"  median |MS - OSM| = {np.median(np.hypot(*(mx - ox).T)):.2f} px")
    for ax, lab in ((0, "dx"), (1, "dy")):
        if mx[:, ax].std() > 1e-9 and ox[:, ax].std() > 1e-9:
            print(f"  corr({lab}) = {np.corrcoef(mx[:, ax], ox[:, ax])[0,1]:+.3f}")

## Look at it

Full resolution before any verdict (`bug-014`). Draw **everything** MS returned,
not a selection. The zoom panel is the one that decides whether these are real
footprints on real roofs.

In [ ]:
show = names[:min(4, len(names))]
fig, axes = plt.subplots(len(show), 3, figsize=(15, 5 * len(show)))
axes = np.atleast_2d(axes)
for row, name in enumerate(show):
    img, grid, meta = load_location(name)
    arr = np.array(img)
    ms, _ = burn(features_for(name), grid, arr.shape[:2])
    ref = osm_mask(name, "building")

    axes[row, 0].imshow(arr)
    axes[row, 0].set_title(f"{name}\nimagery", fontsize=8)

    rgb = np.zeros((*ms.shape, 3))
    rgb[..., 0] = ms            # MS red
    rgb[..., 1] = ref           # OSM green -> agreement is yellow
    axes[row, 1].imshow(arr)
    axes[row, 1].imshow(rgb, alpha=0.55)
    axes[row, 1].set_title("red = MS · green = OSM · yellow = both", fontsize=8)

    ys, xs = np.nonzero(ms | ref)
    if len(ys):
        cy, cx = int(np.median(ys)), int(np.median(xs))
        h = 300
        sl = (slice(max(0, cy - h), cy + h), slice(max(0, cx - h), cx + h))
        axes[row, 2].imshow(arr[sl])
        axes[row, 2].imshow(rgb[sl], alpha=0.55)
    axes[row, 2].set_title("native-resolution zoom", fontsize=8)
    for a in axes[row]:
        a.axis("off")
plt.tight_layout()
plt.savefig(FIGS / "ms-vs-osm.png", dpi=130, bbox_inches="tight")
plt.show()

## Verdict

`worked` / `failed` / `parked`, two or three lines, then copy into `notes.md`.

### What is *not* established